# Selección temporal y reducción de variables

La búsqueda y sus resultados son parte del experimento común. Los parámetros no se copian entre regresión y clasificación.

## Inicialización

Usamos la misma identidad de datos, código y protocolo en todos los cuadernos.

In [1]:
from pathlib import Path
import sys, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd()
while not (ROOT/'scripts/mlcore').is_dir():
    if ROOT == ROOT.parent: raise RuntimeError('No se encontro la raiz del proyecto')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT/'scripts'))
from mlcore.experiment import initialize, aligned
from mlcore.data import read_data, FOLDS, TEST
OUT = initialize()
print('Ejecucion verificada:', OUT.name)


Ejecucion verificada: 01afe756130bafc6


**Qué encontramos y cómo se usa.** La identidad evita mezclar artefactos de protocolos distintos. Los cálculos científicos viven en `scripts/mlcore`; estos cuadernos explican, ejecutan y muestran cada etapa.

## Configuraciones por fold

Cada entrada registra temporadas interiores, pérdidas de la rejilla y configuración elegida.

In [2]:
params = json.loads((OUT/'parameters_by_fold.json').read_text())
display(pd.DataFrame(params).head(25))
assert all(max(p['inner']) < p['season'] for p in params)
print('Configuraciones exteriores registradas:',len(params))

,candidate,season,parameter,inner,scores,chosen
0,con/LR,1718,{'C': 0.05},[1617],"[0.9501565117089803, 0.948352720940106, 0.9484...",1
1,con/LR,1819,{'C': 0.001},"[1617, 1718]","[0.9503049520517572, 0.9500883969828171, 0.950...",0
2,con/LR,1920,{'C': 0.001},"[1718, 1819]","[0.9590130606525873, 0.9588933346128993, 0.959...",0
3,con/LR,2021,{'C': 0.05},"[1819, 1920]","[0.9733270965104606, 0.9723306706177044, 0.972...",1
4,con/LR,2122,{'C': 0.05},"[1920, 2021]","[0.9855892913692639, 0.9841004975590137, 0.984...",1
5,con/LR,2223,{'C': 0.05},"[2021, 2122]","[0.9835757998338167, 0.9809564573247677, 0.981...",1
6,con/LR,2324,{'C': 0.05},"[2122, 2223]","[0.9687404942582953, 0.9681212132992151, 0.968...",1
7,con/HGB,1718,"{'max_depth': 1, 'max_iter': 80, 'learning_rat...",[1617],"[0.9477992747506645, 0.9528840099616287, 0.968...",0
8,con/HGB,1819,"{'max_depth': 1, 'max_iter': 80, 'learning_rat...","[1617, 1718]","[0.9477639623507998, 0.9522079504023535, 0.963...",0
9,con/HGB,1920,"{'max_depth': 1, 'max_iter': 80, 'learning_rat...","[1718, 1819]","[0.9560790109344117, 0.9576047033623583, 0.963...",0


Configuraciones exteriores registradas: 203


**Qué encontramos / decisión.** El SVM calibra después de su entrenamiento base. La MLP usa tres semillas y los árboles no hacen parada temprana con un split aleatorio.

## MI/PCA para ambas ramas

Comparamos las alternativas predefinidas con todas las variables después del preprocesamiento común.

In [3]:
results = pd.read_csv(OUT/'model_comparison.csv')
for branch in ['con','sin']:
    names = [branch+'/LR'+x for x in ['', '_MI5','_MI10','_PCA5','_PCA95']]
    t = results.set_index('candidate').loc[names].copy()
    t['delta_vs_LR'] = t.log_loss-t.iloc[0].log_loss
    display(t[['log_loss','delta_vs_LR','AUC_macro','ECE_masa']])

,log_loss,delta_vs_LR,AUC_macro,ECE_masa
candidate,,,,
con/LR,0.969894,0.000000,0.679279,0.008755
con/LR_MI5,0.967937,-0.001957,0.679784,0.012590
con/LR_MI10,0.968173,-0.001722,0.679814,0.013031
con/LR_PCA5,0.979972,0.010078,0.669333,0.016580
con/LR_PCA95,0.974505,0.004611,0.676080,0.010768


,log_loss,delta_vs_LR,AUC_macro,ECE_masa
candidate,,,,
sin/LR,0.980137,0.000000,0.669522,0.016032
sin/LR_MI5,0.982161,0.002023,0.666636,0.018108
sin/LR_MI10,0.981582,0.001445,0.667251,0.017490
sin/LR_PCA5,0.982406,0.002269,0.667877,0.017624
sin/LR_PCA95,0.980089,-0.000048,0.669988,0.015972


**Qué encontramos / decisión.** La reducción es útil solo si sus probabilidades mejoran. No se fuerza una explicación causal basada únicamente en qué método gana.